# SeaweedMat E09 — External transferability
Reproduction notebook for the locally executed E09. Upload both original Dryad ZIP archives with unchanged names and `outputs/E09/frozen_cotton_parameters.csv` to the Colab working directory. The exact full executed notebook is delivered separately. No model refitting. The two time points are not independent rope validation.


In [ ]:
from pathlib import Path
import zipfile, io, hashlib
import pandas as pd
import numpy as np
ROOT=Path.cwd()
OUT=ROOT/'E09_results'; OUT.mkdir(exist_ok=True)
def locate(name):
    for p in [ROOT/name,ROOT.parent/name,ROOT/'outputs/E09'/name,ROOT.parent/'outputs/E09'/name]:
        if p.exists(): return p
    raise FileNotFoundError(name)
z1=locate('doi_10_5061_dryad_7wm37pvwk__v20220512.zip')
z2=locate('doi_10_5061_dryad_n8274__v20170928.zip')
param=locate('frozen_cotton_parameters.csv')
assert hashlib.sha256(z1.read_bytes()).hexdigest()=='59607bf8f99fb92dfbf936c156c355c6c3ddbb567c5bb2ee8142917d35da6aa8'
assert hashlib.sha256(z2.read_bytes()).hexdigest()=='fa2c4cdfcfd80246bafea5b951e507d180627d7dfd49e18ee4c9522d30a6e542'
with zipfile.ZipFile(z1) as z:
    d=pd.read_csv(z.open('Stream-to-Sea_cotton_strip_data.csv'),sep=';')
assert len(d)==48 and d.nr.is_unique
base=d.loc[d.Treatment.eq('Control'),'tensileStrength'].mean()
g=d.groupby(['Treatment','time']).tensileStrength.agg(n='count',mean='mean',sd='std').reset_index()
g['retention_pct']=100*g['mean']/base
g.to_csv(OUT/'cotton_external_summary.csv',index=False)
obs=g[g.Treatment.eq('Marine')].set_index('time').retention_pct
p=pd.read_csv(param); assert len(p)==35
rows=[]
for r in p.itertuples(index=False):
    for t,y in obs.items():
        x=t/r.time_scale_days; k=r.rate_scaled; s=r.shape_or_lag_scaled
        pred={'constant':lambda:100,'zero_order':lambda:100*max(0,1-k*x),'first_order':lambda:100*np.exp(-k*x),'weibull':lambda:100*np.exp(-k*x**s),'lag_first_order':lambda:100*np.exp(-k*max(0,x-s))}[r.model]()
        rows.append(dict(configuration=r.label,source='Zhou graph' if r.label=='Zhou Cotton' else 'Exact twine',model=r.model,day=t,observed_pct=y,predicted_pct=pred,error_pp=pred-y))
P=pd.DataFrame(rows); assert len(P)==70
P.to_csv(OUT/'frozen_transfer_predictions.csv',index=False)
A=P[P.source.eq('Exact twine')].groupby(['model','day']).agg(observed_pct=('observed_pct','first'),predicted_pct=('predicted_pct','median'),curve_min=('predicted_pct','min'),curve_max=('predicted_pct','max')).reset_index()
A['error_pp']=A.predicted_pct-A.observed_pct
A['within_curve_range']=(A.observed_pct>=A.curve_min)&(A.observed_pct<=A.curve_max)
A.to_csv(OUT/'twine_ensemble_transfer.csv',index=False)
M=A.groupby('model').error_pp.agg(MAE_pp=lambda x:abs(x).mean(),RMSE_pp=lambda x:np.sqrt(np.mean(x*x)),bias_pp='mean').reset_index()
M['n_external_timepoints']=2;M.to_csv(OUT/'transfer_metrics.csv',index=False)
np.testing.assert_allclose(M.RMSE_pp,[32.446593,15.359501,14.128708,16.539246,14.998900],atol=1e-6,rtol=0)
display(g.round(3),A.round(3),M.round(3))
with zipfile.ZipFile(z2) as z:
    f=pd.read_excel(io.BytesIO(z.read(z.namelist()[0])),sheet_name='all_data_only family level')
assert len(f)==100 and f.TP.notna().sum()==79 and f.Disintegration.notna().sum()==80
F=f.groupby(['Type','Habitat','Time']).agg(n_force=('TP','count'),mean_force_N=('TP','mean'),n_area=('Disintegration','count'),area_loss_pct=('Disintegration','mean')).reset_index()
F['retention_pct']=F.groupby(['Type','Habitat']).mean_force_N.transform(lambda s:100*s/s.iloc[0]);F.to_csv(OUT/'plastic_film_boundary.csv',index=False)
display(F.round(3))


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.family':'DejaVu Sans','font.weight':'bold','axes.labelweight':'bold','axes.labelsize':15,'font.size':12,'axes.linewidth':2,'xtick.major.width':2,'ytick.major.width':2,'xtick.major.size':6,'ytick.major.size':6,'axes.spines.top':False,'axes.spines.right':False,'legend.fontsize':10})
fig,ax=plt.subplots(figsize=(155/25.4,115/25.4))
for model,c in [('lag_first_order','#075DE8'),('weibull','#008D79'),('first_order','#C55A11'),('zero_order','#7C3AED')]:
    a=A[A.model.eq(model)];ax.plot(a.day,a.predicted_pct,marker='o',label=model.replace('_',' '),color=c,lw=2)
ax.scatter(obs.index,obs.values,color='black',marker='D',s=65,label='External marine strips',zorder=5)
ax.set_xlabel('Exposure time [days]');ax.set_ylabel('Strength retention [%]');ax.set_xticks([15,41]);ax.legend(frameon=False);fig.tight_layout()
for ext in ['png','svg','pdf']:fig.savefig(OUT/('transfer.'+ext),dpi=600,facecolor='white')
plt.show()
fig,ax=plt.subplots(figsize=(155/25.4,115/25.4))
for (mat,hab),a in F.groupby(['Type','Habitat']):
    ax.plot(a.Time,a.retention_pct,marker='o',label=f'{mat} {hab}',lw=2)
ax.set_xlabel('Exposure time [months]');ax.set_ylabel('Strength retention [%]');ax.legend(frameon=False);fig.tight_layout()
for ext in ['png','svg','pdf']:fig.savefig(OUT/('film_boundary.'+ext),dpi=600,facecolor='white')
plt.show()
